# 03 — CHAIR trên caption Test của hai kho tri thức

**Chạy sau notebook 02** khi đã có `test_predictions.json` cho hai nhánh của ít nhất một seed. Notebook này chỉ **đọc caption đã sinh bằng Beam Search** và bốn file annotation COCO 2014 gốc; không train, không chạy lại mô hình, không dùng GPU. Có thể chạy lại sau khi hoàn tất thêm seed.

CHAIR dùng đúng 80 dòng từ đồng nghĩa trong [`LisaAnne/Hallucination/data/synonyms.txt`](https://github.com/LisaAnne/Hallucination/blob/master/data/synonyms.txt), cách tách từ và singularize theo mã gốc. CHAIR_s là tỷ lệ caption có ít nhất một vật thể COCO không được xác nhận; CHAIR_i là tỷ lệ lượt nhắc vật thể COCO sai trên tổng lượt nhắc vật thể COCO. **Cả hai càng thấp càng tốt.** Vật thể đúng được lấy từ hợp của COCO instance labels và các caption tham chiếu, như [`utils/chair.py`](https://github.com/LisaAnne/Hallucination/blob/master/utils/chair.py). Đây là bản chuyển Python 3 của phép chấm, không dùng một danh sách `coco_synonyms` tự viết.

Lưu ý: CHAIR chỉ đo ảo giác **vật thể thuộc 80 lớp COCO**, không bao phủ sai quan hệ, thuộc tính hay vật thể ngoài bộ từ này. CHAIR_i là tỷ số trên toàn bộ lượt nhắc, không phải trung bình CHAIR_i theo ảnh.


## 1. Gắn Drive và cài bộ đọc COCO / NLTK


In [ ]:
import sys, json, hashlib, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1', 'ijson>=3.3,<4'], check=True)
import nltk
for package in ('punkt', 'punkt_tab'):
    if not nltk.download(package, quiet=True):
        raise RuntimeError('Không tải được NLTK ' + package)
assert nltk.word_tokenize('A man rides a horse.') == ['A', 'man', 'rides', 'a', 'horse', '.']


## 2. Đường dẫn và seed
Cần `instances_train2014.json`, `instances_val2014.json`, `captions_train2014.json`, `captions_val2014.json` trong cùng thư mục annotation. Dữ liệu Karpathy `dataset_coco.json` một mình không đủ cho CHAIR gốc.


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
OUTPUT_ROOT = DATA_ROOT/'baseline_level6/Dung_KhoTriThucLV2'
ANNOTATIONS_DIR = None  # Nếu 4 annotation COCO 2014 nằm chỗ khác, điền Path('...') ở đây.
SEEDS = (42, 44, 46)
if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError('Không thấy thư mục kết quả Dung_KhoTriThucLV2 trên Drive')
print('Đọc dự đoán:', OUTPUT_ROOT/'runs')
print('Lưu CHAIR riêng:', OUTPUT_ROOT/'chair_rohrbach')


## 3. Nạp mã chấm và `synonyms.txt` gốc đã nhúng
Mã tính CHAIR chuyển cú pháp sang Python 3 nhưng giữ công thức, quy tắc cụm từ và phép singularize của bản gốc. Mỗi file được đối chiếu SHA-256 trước khi chạy.


In [ ]:
EMBEDDED_SOURCES = {'chair_score.py': '"""Python 3 CHAIR scorer matching LisaAnne/Hallucination/utils/chair.py.\n\nUses the unmodified upstream data/synonyms.txt and Pattern\'s singularize rules.\nGround truth is the union of COCO 2014 instances and reference-caption objects.\nOnly generated COCO-object mentions enter CHAIR_i; CHAIR_s counts captions.\n"""\nimport csv\nimport hashlib\nimport json\nimport io\nfrom pathlib import Path\nfrom drive_io import read_json, write_json, write_bytes\n\nfrom pattern_singular import singularize\n\nARMS = (\'caption_control\', \'hybrid_3cap_2spo\')\nSEEDS = (42, 44, 46)\nANNOTATION_NAMES = tuple(f\'{kind}_{split}2014.json\'\n                         for kind in (\'instances\', \'captions\') for split in (\'train\', \'val\'))\nDOUBLE_WORDS = [\'motor bike\', \'motor cycle\', \'air plane\', \'traffic light\', \'street light\',\n                \'traffic signal\', \'stop light\', \'fire hydrant\', \'stop sign\', \'parking meter\',\n                \'suit case\', \'sports ball\', \'baseball bat\', \'baseball glove\', \'tennis racket\',\n                \'wine glass\', \'hot dog\', \'cell phone\', \'mobile phone\', \'teddy bear\',\n                \'hair drier\', \'potted plant\', \'bow tie\', \'laptop computer\', \'stove top oven\',\n                \'hot dog\', \'teddy bear\', \'home plate\', \'train track\']\nANIMALS = [\'bird\', \'cat\', \'dog\', \'horse\', \'sheep\', \'cow\', \'elephant\', \'bear\',\n           \'zebra\', \'giraffe\', \'animal\', \'cub\']\nVEHICLES = [\'jet\', \'train\']\n\n\ndef digest(value):\n    body = json.dumps(value, ensure_ascii=False, sort_keys=True,\n                      separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n    return hashlib.sha256(body).hexdigest()\n\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef iter_items(path, prefix):\n    """Stream the large official COCO annotation JSON where ijson is available."""\n    try:\n        import ijson\n    except ImportError:\n        for item in read_json(path)[prefix]:\n            yield item\n    else:\n        with Path(path).open(\'rb\') as stream:\n            yield from ijson.items(stream, prefix + \'.item\')\n\n\ndef annotation_dir(data_root, override=None):\n    root = Path(data_root)\n    candidates = ([Path(override)] if override is not None else [\n        root/\'COCO_2014/annotations_trainval2014/annotations\',\n        root/\'COCO_2014/annotations\', root/\'annotations\', root])\n    matches = [p for p in candidates if all((p/name).is_file() for name in ANNOTATION_NAMES)]\n    if len(matches) != 1:\n        raise FileNotFoundError(\'CHAIR cần đúng 4 file COCO 2014 gốc: \' +\n            \', \'.join(ANNOTATION_NAMES) + \'. Đặt ANNOTATIONS_DIR nếu chúng ở nơi khác.\')\n    return matches[0]\n\n\nclass RohrbachCHAIR:\n    def __init__(self, synonyms_path, tokenizer=None):\n        if tokenizer is None:\n            import nltk\n            tokenizer = nltk.word_tokenize\n        self.tokenizer = tokenizer\n        lines = Path(synonyms_path).read_text(encoding=\'utf-8\').splitlines()\n        if len(lines) != 80:\n            raise ValueError(\'Expected 80 original COCO synonym rows\')\n        # Preserve the upstream comma-space split, including its duplicate entries.\n        synonyms = [line.strip().split(\', \') for line in lines]\n        self.mscoco_objects = set(word for row in synonyms for word in row)\n        self.inverse_synonym_dict = {word: row[0] for row in synonyms for word in row}\n        self.double_word_dict = {word: word for word in DOUBLE_WORDS}\n        for animal in ANIMALS:\n            self.double_word_dict[\'baby \' + animal] = animal\n            self.double_word_dict[\'adult \' + animal] = animal\n        for vehicle in VEHICLES:\n            self.double_word_dict[\'passenger \' + vehicle] = vehicle\n        self.double_word_dict.update({\'bow tie\': \'tie\', \'toilet seat\': \'toilet\',\n                                      \'wine glas\': \'wine glass\'})\n        self.synonyms_sha256 = file_sha256(synonyms_path)\n\n    def caption_to_words(self, caption):\n        # Original order: NLTK tokenizer, Pattern singularize, then COCO phrase rules.\n        words = [singularize(word) for word in self.tokenizer(caption.lower())]\n        collapsed, positions = [], []\n        i = 0\n        while i < len(words):\n            positions.append(i)\n            phrase = \' \'.join(words[i:i+2])\n            if phrase in self.double_word_dict:\n                collapsed.append(self.double_word_dict[phrase])\n                i += 2\n            else:\n                collapsed.append(words[i])\n                i += 1\n        if \'toilet\' in collapsed and \'seat\' in collapsed:\n            collapsed = [word for word in collapsed if word != \'seat\']\n        # Original implementation indexes positions after the toilet-seat filter.\n        selected = [(word, positions[j]) for j, word in enumerate(collapsed)\n                    if word in self.mscoco_objects]\n        return [(word, self.inverse_synonym_dict[word], index)\n                for word, index in selected]\n\n    def ground_truth(self, ann_dir, image_ids):\n        ids = set(image_ids)\n        objects = {iid: set() for iid in ids}\n        # Rohrbach first reads segmentation categories, then reference captions.\n        for split in (\'val\', \'train\'):\n            path = Path(ann_dir)/f\'instances_{split}2014.json\'\n            categories = {int(row[\'id\']): row[\'name\'] for row in iter_items(path, \'categories\')}\n            for row in iter_items(path, \'annotations\'):\n                iid = int(row[\'image_id\'])\n                if iid in ids:\n                    name = categories[int(row[\'category_id\'])]\n                    if name not in self.inverse_synonym_dict:\n                        raise ValueError(\'COCO category absent from official synonyms: \' + name)\n                    objects[iid].add(self.inverse_synonym_dict[name])\n        for split in (\'val\', \'train\'):\n            path = Path(ann_dir)/f\'captions_{split}2014.json\'\n            for row in iter_items(path, \'annotations\'):\n                iid = int(row[\'image_id\'])\n                if iid in ids:\n                    objects[iid].update(node for _, node, _ in\n                                        self.caption_to_words(row[\'caption\']))\n        return objects\n\n    def score(self, predictions, objects):\n        if not predictions:\n            raise ValueError(\'Empty prediction set\')\n        image_ids = [row[\'image_id\'] for row in predictions]\n        if len(set(image_ids)) != len(image_ids) or set(image_ids) != set(objects):\n            raise ValueError(\'Prediction image IDs mismatch ground-truth image IDs\')\n        count_mentions = count_hallucinated = count_sentences = 0\n        examples = []\n        for row in predictions:\n            mentions = self.caption_to_words(row[\'caption\'])\n            wrong = [(word, node, idx) for word, node, idx in mentions\n                     if node not in objects[row[\'image_id\']]]\n            count_mentions += len(mentions)\n            count_hallucinated += len(wrong)\n            count_sentences += bool(wrong)\n            if wrong and len(examples) < 50:\n                examples.append(dict(image_id=row[\'image_id\'], caption=row[\'caption\'],\n                                     hallucinated_objects=[node for _, node, _ in wrong],\n                                     gt_objects=sorted(objects[row[\'image_id\']])))\n        if count_mentions == 0:\n            raise ValueError(\'No generated COCO object mentions: CHAIR_i undefined\')\n        return dict(CHAIR_s=count_sentences/len(predictions),\n                    CHAIR_i=count_hallucinated/count_mentions,\n                    captions=len(predictions), hallucinated_captions=count_sentences,\n                    object_mentions=count_mentions,\n                    hallucinated_object_mentions=count_hallucinated,\n                    examples=examples)\n\n\ndef load_prediction_pair(output_root, seed):\n    root = Path(output_root)/\'runs\'/f\'seed{seed}\'\n    if not (root/\'test_plan.json\').is_file():\n        raise FileNotFoundError(f\'Incomplete Test plan for seed {seed}\')\n    plan = read_json(root/\'test_plan.json\')\n    if set(plan.get(\'models\', {})) != set(ARMS):\n        raise ValueError(f\'Incomplete frozen Test models for seed {seed}\')\n    rows = {}\n    for arm in ARMS:\n        folder = root/arm\n        summary = read_json(folder/\'test_summary.json\')\n        evaluation = summary[\'evaluation\']\n        if (not summary.get(\'complete\') or evaluation.get(\'split\') != \'test\' or\n                evaluation.get(\'seed\') != seed or evaluation.get(\'arm\') != arm or\n                evaluation.get(\'study\') != plan.get(\'study\') or\n                evaluation.get(\'model_state_sha256\') != plan[\'models\'][arm] or\n                evaluation.get(\'beam\') != 3 or evaluation.get(\'max_len\') != 30 or\n                evaluation.get(\'alpha\') != 0.6 or\n                read_json(folder/\'test_once.json\') != evaluation):\n            raise ValueError(f\'Test Beam contract mismatch: seed {seed}, {arm}\')\n        if summary.get(\'summary_sha256\') != digest({k: v for k, v in summary.items()\n                                                    if k != \'summary_sha256\'}):\n            raise ValueError(f\'Test summary checksum mismatch: seed {seed}, {arm}\')\n        preds = read_json(folder/\'test_predictions.json\')\n        ids = [r[\'image_id\'] for r in preds]\n        if (len(ids) != 5000 or len(set(ids)) != 5000 or\n                any(type(i) is not int or type(r.get(\'caption\')) is not str for i, r in zip(ids, preds)) or\n                digest(ids) != evaluation[\'ids_sha256\']):\n            raise ValueError(f\'Invalid Karpathy Test predictions: seed {seed}, {arm}\')\n        rows[arm] = (preds, summary, file_sha256(folder/\'test_predictions.json\'))\n    if [r[\'image_id\'] for r in rows[ARMS[0]][0]] != [r[\'image_id\'] for r in rows[ARMS[1]][0]]:\n        raise ValueError(\'Paired Test image order differs\')\n    return rows\n\n\ndef run(data_root, output_root, synonyms_path, annotations_override=None, seeds=SEEDS):\n    ann = annotation_dir(data_root, annotations_override)\n    scorer = RohrbachCHAIR(synonyms_path)\n    available = {}\n    for seed in seeds:\n        try:\n            available[seed] = load_prediction_pair(output_root, seed)\n        except FileNotFoundError as exc:\n            print(\'Chưa chấm seed\', seed, \':\', exc)\n    if not available:\n        raise FileNotFoundError(\'Chưa có cặp Test Beam hoàn tất ở bất kỳ seed nào\')\n    all_ids = {r[\'image_id\'] for pair in available.values() for arm in ARMS for r in pair[arm][0]}\n    gt = scorer.ground_truth(ann, all_ids)\n    annotation_hashes = {name: file_sha256(ann/name) for name in ANNOTATION_NAMES}\n    source_hashes = {name: file_sha256(Path(__file__).with_name(name))\n                     for name in (\'chair_score.py\', \'pattern_singular.py\')}\n    out = Path(output_root)/\'chair_rohrbach\'\n    rows = []\n    for seed, pair in available.items():\n        for arm in ARMS:\n            predictions, summary, pred_sha = pair[arm]\n            result = scorer.score(predictions, gt)\n            identity = dict(source_repo=\'https://github.com/LisaAnne/Hallucination\',\n                source_file=\'utils/chair.py\', synonyms_file=\'data/synonyms.txt\',\n                synonyms_sha256=scorer.synonyms_sha256, predictions_sha256=pred_sha,\n                test_summary_sha256=file_sha256(Path(output_root)/\'runs\'/f\'seed{seed}\'/arm/\'test_summary.json\'),\n                annotation_paths={name: str(ann/name) for name in ANNOTATION_NAMES},\n                annotation_sha256=annotation_hashes, scorer_source_sha256=source_hashes,\n                normalization=\'nltk.word_tokenize + clips/pattern singularize + original CHAIR phrase rules\',\n                ground_truth=\'COCO 2014 instances union reference captions\',\n                decoding=\'Beam search, beam=3, max_len=30, alpha=0.6\')\n            destination = out/f\'seed{seed}\'/arm/\'chair_summary.json\'\n            payload = dict(seed=seed, arm=arm, identity=identity, **result)\n            if destination.is_file() and read_json(destination)[\'identity\'] != identity:\n                raise ValueError(\'Existing CHAIR output has different inputs: \' + str(destination))\n            write_json(destination, payload)\n            rows.append(dict(seed=seed, arm=arm, CHAIR_s=result[\'CHAIR_s\']*100,\n                             CHAIR_i=result[\'CHAIR_i\']*100,\n                             hallucinated_captions=result[\'hallucinated_captions\'],\n                             hallucinated_object_mentions=result[\'hallucinated_object_mentions\'],\n                             object_mentions=result[\'object_mentions\']))\n    out.mkdir(parents=True, exist_ok=True)\n    csv_path = out/\'test_chair_by_seed.csv\'\n    stream = io.StringIO(newline=\'\')\n    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n    writer.writeheader(); writer.writerows(rows)\n    write_bytes(csv_path, stream.getvalue().encode(\'utf-8-sig\'))\n    if set(available) == set(SEEDS):\n        import statistics\n        aggregate = []\n        for arm in ARMS:\n            subset = [r for r in rows if r[\'arm\'] == arm]\n            aggregate.append(dict(arm=arm, seeds=list(SEEDS),\n                CHAIR_s_mean=statistics.mean(r[\'CHAIR_s\'] for r in subset),\n                CHAIR_s_std=statistics.stdev(r[\'CHAIR_s\'] for r in subset),\n                CHAIR_i_mean=statistics.mean(r[\'CHAIR_i\'] for r in subset),\n                CHAIR_i_std=statistics.stdev(r[\'CHAIR_i\'] for r in subset)))\n        deltas = [{metric: next(r[metric] for r in rows if r[\'seed\'] == seed and r[\'arm\'] == ARMS[1]) -\n                           next(r[metric] for r in rows if r[\'seed\'] == seed and r[\'arm\'] == ARMS[0])\n                   for metric in (\'CHAIR_s\', \'CHAIR_i\')} for seed in SEEDS]\n        aggregate.append(dict(arm=\'delta_hybrid_minus_control\', seeds=list(SEEDS),\n            CHAIR_s_mean=statistics.mean(r[\'CHAIR_s\'] for r in deltas),\n            CHAIR_s_std=statistics.stdev(r[\'CHAIR_s\'] for r in deltas),\n            CHAIR_i_mean=statistics.mean(r[\'CHAIR_i\'] for r in deltas),\n            CHAIR_i_std=statistics.stdev(r[\'CHAIR_i\'] for r in deltas)))\n        write_json(out/\'test_chair_3seed_summary.json\', aggregate)\n    return rows\n', 'pattern_singular.py': '# Extracted without logic changes from clips/pattern pattern/text/en/inflect.py,\n# singularize() and its rule tables (see upstream license and attribution).\n# https://github.com/clips/pattern/blob/master/pattern/text/en/inflect.py\nimport re\nNOUN = \'NN\'\nplural_prepositions = set(\'about before during of till above behind except off to across below for on under after beneath from onto until among beside in out unto around besides into over upon at between near since with athwart betwixt beyond but by\'.split())\n#### SINGULARIZE ###################################################################################\n# Adapted from Bermi Ferrer\'s Inflector for Python:\n# http://www.bermi.org/inflector/\n# Copyright (c) 2006 Bermi Ferrer Martinez\n# Permission is hereby granted, free of charge, to any person obtaining a copy\n# of this software to deal in this software without restriction, including\n# without limitation the rights to use, copy, modify, merge, publish,\n# distribute, sublicense, and/or sell copies of this software, and to permit\n# persons to whom this software is furnished to do so, subject to the following\n# condition:\n#\n# THIS SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\n# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\n# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\n# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\n# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\n# OUT OF OR IN CONNECTION WITH THIS SOFTWARE OR THE USE OR OTHER DEALINGS IN\n# THIS SOFTWARE.\nsingular_rules = [\n    (r\'(?i)(.)ae$\'            , \'\\\\1a\'    ),\n    (r\'(?i)(.)itis$\'          , \'\\\\1itis\' ),\n    (r\'(?i)(.)eaux$\'          , \'\\\\1eau\'  ),\n    (r\'(?i)(quiz)zes$\'        , \'\\\\1\'     ),\n    (r\'(?i)(matr)ices$\'       , \'\\\\1ix\'   ),\n    (r\'(?i)(ap|vert|ind)ices$\', \'\\\\1ex\'   ),\n    (r\'(?i)^(ox)en\'           , \'\\\\1\'     ),\n    (r\'(?i)(alias|status)es$\' , \'\\\\1\'     ),\n    (r\'(?i)([octop|vir])i$\'   , \'\\\\1us\'  ),\n    (r\'(?i)(cris|ax|test)es$\' , \'\\\\1is\'   ),\n    (r\'(?i)(shoe)s$\'          , \'\\\\1\'     ),\n    (r\'(?i)(o)es$\'            , \'\\\\1\'     ),\n    (r\'(?i)(bus)es$\'          , \'\\\\1\'     ),\n    (r\'(?i)([m|l])ice$\'       , \'\\\\1ouse\' ),\n    (r\'(?i)(x|ch|ss|sh)es$\'   , \'\\\\1\'     ),\n    (r\'(?i)(m)ovies$\'         , \'\\\\1ovie\' ),\n    (r\'(?i)(.)ombies$\'        , \'\\\\1ombie\'),\n    (r\'(?i)(s)eries$\'         , \'\\\\1eries\'),\n    (r\'(?i)([^aeiouy]|qu)ies$\', \'\\\\1y\'    ),\n        # -f, -fe sometimes take -ves in the plural\n        # (e.g., lives, wolves).\n    (r"([aeo]l)ves$"          , "\\\\1f"    ),\n    (r"([^d]ea)ves$"          , "\\\\1f"    ),\n    (r"arves$"                , "arf"     ),\n    (r"erves$"                , "erve"    ),\n    (r"([nlw]i)ves$"          , "\\\\1fe"   ),\n    (r\'(?i)([lr])ves$\'        , \'\\\\1f\'    ),\n    (r"([aeo])ves$"           , "\\\\1ve"   ),\n    (r\'(?i)(sive)s$\'          , \'\\\\1\'     ),\n    (r\'(?i)(tive)s$\'          , \'\\\\1\'     ),\n    (r\'(?i)(hive)s$\'          , \'\\\\1\'     ),\n    (r\'(?i)([^f])ves$\'        , \'\\\\1fe\'   ),\n    # -ses suffixes.\n    (r\'(?i)(^analy)ses$\'      , \'\\\\1sis\'  ),\n    (r\'(?i)((a)naly|(b)a|(d)iagno|(p)arenthe|(p)rogno|(s)ynop|(t)he)ses$\', \'\\\\1\\\\2sis\'),\n    (r\'(?i)(.)opses$\'         , \'\\\\1opsis\'),\n    (r\'(?i)(.)yses$\'          , \'\\\\1ysis\' ),\n    (r\'(?i)(h|d|r|o|n|b|cl|p)oses$\', \'\\\\1ose\'),\n    (r\'(?i)(fruct|gluc|galact|lact|ket|malt|rib|sacchar|cellul)ose$\', \'\\\\1ose\'),\n    (r\'(?i)(.)oses$\'          , \'\\\\1osis\' ),\n    # -a\n    (r\'(?i)([ti])a$\'          , \'\\\\1um\'   ),\n    (r\'(?i)(n)ews$\'           , \'\\\\1ews\'  ),\n    (r\'(?i)s$\'                , \'\'        ),\n]\n# For performance, compile the regular expressions only once:\nsingular_rules = [(re.compile(r[0]), r[1]) for r in singular_rules]\nsingular_uninflected = set((\n    "bison"      , "debris"   , "headquarters", "pincers"    , "trout"     ,\n    "bream"      , "diabetes" , "herpes"      , "pliers"     , "tuna"      ,\n    "breeches"   , "djinn"    , "high-jinks"  , "proceedings", "whiting"   ,\n    "britches"   , "eland"    , "homework"    , "rabies"     , "wildebeest",\n    "carp"       , "elk"      , "innings"     , "salmon"     ,\n    "chassis"    , "flounder" , "jackanapes"  , "scissors"   ,\n    "christmas"  , "gallows"  , "mackerel"    , "series"     ,\n    "clippers"   , "georgia"  , "measles"     , "shears"     ,\n    "cod"        , "graffiti" , "mews"        , "species"    ,\n    "contretemps",              "mumps"       , "swine"      ,\n    "corps"      ,              "news"        , "swiss"      ,\n))\nsingular_uncountable = set((\n    "advice"     , "equipment", "happiness"   , "luggage"    , "news"      , "software"     ,\n    "bread"      , "fruit"    , "information" , "mathematics", "progress"  , "understanding",\n    "butter"     , "furniture", "ketchup"     , "mayonnaise" , "research"  , "water"        ,\n    "cheese"     , "garbage"  , "knowledge"   , "meat"       , "rice"      ,\n    "electricity", "gravel"   , "love"        , "mustard"    , "sand"      ,\n))\nsingular_ie = set((\n    "alergie"    , "cutie"    , "hoagie"      , "newbie"     , "softie"    , "veggie"       ,\n    "auntie"     , "doggie"   , "hottie"      , "nightie"    , "sortie"    , "weenie"       ,\n    "beanie"     , "eyrie"    , "indie"       , "oldie"      , "stoolie"   , "yuppie"       ,\n    "birdie"     , "freebie"  , "junkie"      , "^pie"       , "sweetie"   , "zombie"       ,\n    "bogie"      , "goonie"   , "laddie"      , "pixie"      , "techie"    ,\n    "bombie"     , "groupie"  , "laramie"     , "quickie"    , "^tie"      ,\n    "collie"     , "hankie"   , "lingerie"    , "reverie"    , "toughie"   ,\n    "cookie"     , "hippie"   , "meanie"      , "rookie"     , "valkyrie"  ,\n))\nsingular_irregular = {\n       "atlantes": "atlas",\n        "atlases": "atlas",\n           "axes": "axe",\n         "beeves": "beef",\n       "brethren": "brother",\n       "children": "child",\n        "corpora": "corpus",\n       "corpuses": "corpus",\n    "ephemerides": "ephemeris",\n           "feet": "foot",\n        "ganglia": "ganglion",\n          "geese": "goose",\n         "genera": "genus",\n          "genii": "genie",\n       "graffiti": "graffito",\n         "helves": "helve",\n           "kine": "cow",\n         "leaves": "leaf",\n         "loaves": "loaf",\n            "men": "man",\n      "mongooses": "mongoose",\n         "monies": "money",\n          "moves": "move",\n         "mythoi": "mythos",\n         "numena": "numen",\n       "occipita": "occiput",\n      "octopodes": "octopus",\n          "opera": "opus",\n         "opuses": "opus",\n            "our": "my",\n           "oxen": "ox",\n          "penes": "penis",\n        "penises": "penis",\n         "people": "person",\n          "sexes": "sex",\n    "soliloquies": "soliloquy",\n          "teeth": "tooth",\n         "testes": "testis",\n        "trilbys": "trilby",\n         "turves": "turf",\n            "zoa": "zoon",\n}\n\ndef singularize(word, pos=NOUN, custom={}):\n    """ Returns the singular of a given word.\n    """\n    if word in custom:\n        return custom[word]\n    # Recurse compound words (e.g. mothers-in-law).\n    if "-" in word:\n        w = word.split("-")\n        if len(w) > 1 and w[1] in plural_prepositions:\n            return singularize(w[0], pos, custom) + "-" + "-".join(w[1:])\n    # dogs\' => dog\'s\n    if word.endswith("\'"):\n        return singularize(word[:-1]) + "\'s"\n    w = word.lower()\n    for x in singular_uninflected:\n        if x.endswith(w):\n            return word\n    for x in singular_uncountable:\n        if x.endswith(w):\n            return word\n    for x in singular_ie:\n        if w.endswith(x + "s"):\n            return w\n    for x in singular_irregular:\n        if w.endswith(x):\n            return re.sub(\'(?i)\' + x + \'$\', singular_irregular[x], word)\n    for suffix, inflection in singular_rules:\n        m = suffix.search(word)\n        g = m and m.groups() or []\n        if m:\n            for k in range(len(g)):\n                if g[k] is None:\n                    inflection = inflection.replace(\'\\\\\' + str(k + 1), \'\')\n            return suffix.sub(inflection, word)\n    return word\n', 'synonyms.txt': 'person, girl, boy, man, woman, kid, child, chef, baker, people, adult, rider, children, baby, worker, passenger, sister, biker, policeman, cop, officer, lady, cowboy, bride, groom, male, female, guy, traveler, mother, father, gentleman, pitcher, player, skier, snowboarder, skater, skateboarder, person, woman, guy, foreigner, child, gentleman, caller, offender, coworker, trespasser, patient, politician, soldier, grandchild, serviceman, walker, drinker, doctor, bicyclist, thief, buyer, teenager, student, camper, driver, solider, hunter, shopper, villager\nbicycle, bike, bicycle, bike, unicycle, minibike, trike\ncar, automobile, van, minivan, sedan, suv, hatchback, cab, jeep, coupe, taxicab, limo, taxi\nmotorcycle, scooter,  motor bike, motor cycle, motorbike, scooter, moped\nairplane, jetliner, plane, air plane, monoplane, aircraft, jet, jetliner, airbus, biplane, seaplane\nbus, minibus, trolley\ntrain, locomotive, tramway, caboose\ntruck, pickup, lorry, hauler, firetruck\nboat, ship, liner, sailboat, motorboat, dinghy, powerboat, speedboat, canoe, skiff, yacht, kayak, catamaran, pontoon, houseboat, vessel, rowboat, trawler, ferryboat, watercraft, tugboat, schooner, barge, ferry, sailboard, paddleboat, lifeboat, freighter, steamboat, riverboat, battleship, steamship\ntraffic light, street light, traffic signal, stop light, streetlight, stoplight\nfire hydrant, hydrant\nstop sign\nparking meter\nbench, pew\nbird, ostrich, owl, seagull, goose, duck, parakeet, falcon, robin, pelican, waterfowl, heron, hummingbird, mallard, finch, pigeon, sparrow, seabird, osprey, blackbird, fowl, shorebird, woodpecker, egret, chickadee, quail, bluebird, kingfisher, buzzard, willet, gull, swan, bluejay, flamingo, cormorant, parrot, loon, gosling, waterbird, pheasant, rooster, sandpiper, crow, raven, turkey, oriole, cowbird, warbler, magpie, peacock, cockatiel, lorikeet, puffin, vulture, condor, macaw, peafowl, cockatoo, songbird\ncat, kitten, feline, tabby\ndog, puppy, beagle, pup, chihuahua, schnauzer, dachshund, rottweiler, canine, pitbull, collie, pug, terrier, poodle, labrador, doggie, doberman, mutt, doggy, spaniel, bulldog, sheepdog, weimaraner, corgi, cocker, greyhound, retriever, brindle, hound, whippet, husky\nhorse, colt, pony, racehorse, stallion, equine, mare, foal, palomino, mustang, clydesdale, bronc, bronco\nsheep, lamb, ram, lamb, goat, ewe\ncow, cattle, oxen, ox, calf, cattle, holstein, heifer, buffalo, bull, zebu, bison\nelephant\nbear, panda\nzebra\ngiraffe\nbackpack, knapsack\numbrella\nhandbag, wallet, purse, briefcase\ntie, bow, bow tie\nsuitcase, suit case, luggage\nfrisbee\nskis, ski\nsnowboard\nsports ball, ball\nkite\nbaseball bat\nbaseball glove\nskateboard\nsurfboard, longboard, skimboard, shortboard, wakeboard\ntennis racket, racket\nbottle\nwine glass\ncup\nfork\nknife, pocketknife, knive\nspoon\nbowl, container\nbanana\napple\nsandwich, burger, sub, cheeseburger, hamburger\norange\nbroccoli\ncarrot\nhot dog\npizza\ndonut, doughnut, bagel\ncake,  cheesecake, cupcake, shortcake, coffeecake, pancake\nchair, seat, stool\ncouch, sofa, recliner, futon, loveseat, settee, chesterfield\npotted plant, houseplant\nbed\ndining table, table, desk\ntoilet, urinal, commode, toilet, lavatory, potty\ntv, monitor, televison, television\nlaptop, computer, notebook, netbook, lenovo, macbook, laptop computer\nmouse\nremote\nkeyboard\ncell phone, mobile phone, phone, cellphone, telephone, phon, smartphone, iPhone\nmicrowave\noven, stovetop, stove, stove top oven\ntoaster\nsink\nrefrigerator, fridge, fridge, freezer\nbook\nclock\nvase\nscissors\nteddy bear, teddybear\nhair drier, hairdryer\ntoothbrush\n', 'LICENSE_PATTERN.txt': 'Copyright (c) 2011-2013 University of Antwerp, Belgium\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n  * Redistributions of source code must retain the above copyright\n    notice, this list of conditions and the following disclaimer.\n  * Redistributions in binary form must reproduce the above copyright\n    notice, this list of conditions and the following disclaimer in\n    the documentation and/or other materials provided with the\n    distribution.\n  * Neither the name of Pattern nor the names of its\n    contributors may be used to endorse or promote products\n    derived from this software without specific prior written\n    permission.\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS\n"AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT\nLIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS\nFOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE\nCOPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT,\nINCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING,\nBUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT\nLIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN\nANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE\nPOSSIBILITY OF SUCH DAMAGE.\n', 'drive_io.py': '"""Verified publication for Colab DriveFS; never depends on remote rename.\n\nStage bytes on local disk, copy/read back a recovery copy, then the destination.\nThis is NOT an atomic transaction or a lock across Colab machines. One writer\nper bank/seed is required. Training checkpoints keep their existing journal.\n"""\nimport errno\nimport hashlib\nimport json\nimport os\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef publish(source, target, attempts=4):\n    """Copy from stable local source; retry transient I/O, verify every byte."""\n    source, target = Path(source), Path(target)\n    expected = sha(source)\n    for attempt in range(attempts):\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            try:\n                dest = target.open(\'r+b\')\n            except FileNotFoundError:\n                dest = target.open(\'xb\')\n            with source.open(\'rb\') as inp, dest:\n                dest.seek(0)\n                for block in iter(lambda: inp.read(8 * 1024 * 1024), b\'\'):\n                    dest.write(block)\n                dest.truncate()\n                dest.flush()\n                try:\n                    os.fsync(dest.fileno())\n                except OSError as exc:\n                    if exc.errno not in {errno.EINVAL, errno.ENOTSUP, errno.ENOSYS}:\n                        raise\n            if sha(target) != expected:\n                raise OSError(errno.EIO, \'Read-back checksum mismatch\', str(target))\n            return expected\n        except OSError as exc:\n            if exc.errno not in {errno.ENOENT, errno.EIO, errno.EAGAIN, errno.EEXIST,\n                                 errno.EBUSY, errno.ETIMEDOUT} or attempt == attempts - 1:\n                raise OSError(f\'Could not publish/verify {target}. Check Drive mount, \'\n                              \'free space and other writers; keep existing outputs.\') from exc\n            time.sleep(.5 * (attempt + 1))\n\n\ndef write_bytes(path, data):\n    path = Path(path)\n    with tempfile.TemporaryDirectory(prefix=\'lv6_publish_\') as folder:\n        local = Path(folder)/\'payload\'\n        local.write_bytes(data)\n        publish(local, path.with_name(path.name + \'.pending\'))\n        publish(local, path)\n\n\ndef write_json(path, value):\n    raw = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode(\'utf-8\')\n    write_bytes(path, raw)\n\n\ndef read_json(path):\n    path = Path(path)\n    try:\n        return json.loads(path.read_text(encoding=\'utf-8\'))\n    except (FileNotFoundError, UnicodeError, json.JSONDecodeError) as original:\n        # Read-only fallback: a later verified write can republish the final file.\n        for suffix in (\'.pending\', \'.tmp\'):\n            try:\n                return json.loads(path.with_name(path.name + suffix).read_text(encoding=\'utf-8\'))\n            except (FileNotFoundError, UnicodeError, json.JSONDecodeError):\n                pass\n        raise original\n\n\ndef json_available(path):\n    path = Path(path)\n    return any(path.with_name(path.name + s).is_file() for s in (\'\', \'.pending\', \'.tmp\'))\n\n\ndef check_output(folder):\n    """Small actual write/read check before expensive Colab work; keep a receipt."""\n    import uuid\n    folder = Path(folder)\n    receipt = folder/\'_io_checks\'/(uuid.uuid4().hex + \'.json\')\n    value = {\'method\': \'local_stage_copy_sha256_v1\', \'folder\': str(folder)}\n    write_json(receipt, value)\n    if read_json(receipt) != value:\n        raise IOError(\'Output preflight failed: \' + str(folder))\n    print(\'Drive write/read OK:\', folder)\n'}
EXPECTED_HASHES = {'chair_score.py': '3172c0c88f7e3910de28d3da8b65ace254c89913bd2438502a4b7a1e7c0a1cdf', 'pattern_singular.py': 'bc4d883f7afc8d3c775af070b70e27c887ad12d71abcb4d3fdbd57aa3001b273', 'synonyms.txt': '84e0040eec10ce42db1f958b9008d78b4f5ec0b7ce702132040db65ff1f48c34', 'LICENSE_PATTERN.txt': '0e4b6c8a0d4d93ef0d25bfed6d945e1e84147ee6d8cc234ba8d6e002bc944a2f', 'drive_io.py': 'f70c35d4503a6fce8d42b8446650d3cb4af655983dd43a8860655609068eec10'}
RUNTIME_DIR = Path('/content/lv6_chair_rohrbach')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for name, content in EMBEDDED_SOURCES.items():
    target = RUNTIME_DIR/name
    target.write_text(content, encoding='utf-8')
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != EXPECTED_HASHES[name]:
        raise RuntimeError('Mã hoặc từ điển CHAIR không khớp: ' + name)
sys.path.insert(0, str(RUNTIME_DIR))
for name in ('drive_io', 'chair_score', 'pattern_singular'):
    sys.modules.pop(name, None)
from chair_score import annotation_dir, run
from drive_io import check_output
source_dir = annotation_dir(DATA_ROOT, ANNOTATIONS_DIR)
check_output(OUTPUT_ROOT/'chair_rohrbach')
print('COCO annotations:', source_dir)
print('Rohrbach synonyms SHA-256:', EXPECTED_HASHES['synonyms.txt'])


## 4. Chấm CHAIR Test Beam trên các seed đã hoàn tất
Chỉ đọc `test_predictions.json` đã khóa của notebook 02. Nếu một seed chưa hoàn tất cả hai nhánh, notebook bỏ qua seed đó. Khi đủ ba seed, xuất thêm mean ± sample std.


In [ ]:
rows = run(DATA_ROOT, OUTPUT_ROOT, RUNTIME_DIR/'synonyms.txt',
           annotations_override=ANNOTATIONS_DIR, seeds=SEEDS)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(rows).round(4))
print('Bảng theo seed:', OUTPUT_ROOT/'chair_rohrbach/test_chair_by_seed.csv')
if (OUTPUT_ROOT/'chair_rohrbach/test_chair_3seed_summary.json').is_file():
    display(pd.DataFrame(json.loads((OUTPUT_ROOT/'chair_rohrbach/test_chair_3seed_summary.json').read_text(encoding='utf-8'))).round(4))


## 5. Đặt CHAIR cạnh 8 chỉ số COCO Test
Tất cả là điểm Test của cùng caption Beam. CHAIR thấp hơn là tốt hơn; 8 chỉ số chất lượng cao hơn thường là tốt hơn. Không chọn lại checkpoint hay siêu tham số dựa trên bảng Test này.


In [ ]:
comparison = []
for row in rows:
    seed, arm = row['seed'], row['arm']
    old = json.loads((OUTPUT_ROOT/'runs'/f'seed{seed}'/arm/'test_summary.json').read_text(encoding='utf-8'))
    comparison.append({'seed': seed, 'arm': arm,
        **{k: v*100 for k, v in old['metrics_raw'].items()},
        'CHAIR_s': row['CHAIR_s'], 'CHAIR_i': row['CHAIR_i']})
display(pd.DataFrame(comparison).round(4))
print('Kết quả chi tiết mỗi nhánh:', OUTPUT_ROOT/'chair_rohrbach')
